# Base Analytics — Validação

Notebook de exploração e validação da base sintética. A geração dos dados fica centralizada em `src/data_generator.py`.

Todos os caminhos são relativos ao projeto; nenhum caminho local de usuário ou empresa é utilizado.

## 1. Carregamento

A base consolidada é lida diretamente do Parquet gerado pelo script.

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
ARQUIVO_PARQUET = ROOT / "data" / "BASE_ANALYTICS_MOCK.parquet"

df = pd.read_parquet(ARQUIVO_PARQUET)
df.head()

## 2. Estrutura e qualidade dos dados

In [ ]:
print(f"Registros: {len(df):,}")
print(f"Colunas: {len(df.columns)}")
print(f"Duplicidades em NUMERO_CONTA: {df['NUMERO_CONTA'].duplicated().sum()}")
print(f"Duplicidades em CPF: {df['CPF'].duplicated().sum()}")
df.info()

## 3. Indicadores da carteira

In [ ]:
resumo = pd.DataFrame({
    "Indicador": ["Contas", "Carteira", "PDD", "Ticket médio", "PDD rate"],
    "Valor": [
        df["NUMERO_CONTA"].nunique(),
        df["SALDO"].sum(),
        df["SALDO_PDD"].sum(),
        df["SALDO"].mean(),
        df["SALDO_PDD"].sum() / df["SALDO"].sum()
    ]
})
resumo

## 4. Distribuições

In [ ]:
display(df["RISCO"].value_counts().rename_axis("RISCO").to_frame("CONTAS"))
display(df["FX_ATRASO"].value_counts().rename_axis("FX_ATRASO").to_frame("CONTAS"))
display(df.groupby("CANAL", dropna=False)["SALDO"].sum().sort_values(ascending=False).to_frame("SALDO"))

## 5. Validações

In [ ]:
assert len(df) == 5000
assert len(df.columns) == 25
assert df["NUMERO_CONTA"].is_unique
assert df["CPF"].is_unique
assert df["SALDO"].ge(0).all()
assert df["SALDO_PDD"].ge(0).all()
assert df["ATRASO"].ge(0).all()
assert df.loc[df["RECUPERADO"] == "SIM", "DIA_RECUPERACAO"].notna().all()
print("✓ Todas as validações foram concluídas com sucesso.")